In [ ]:
#Import all necessary libraries
import json
import re
import numpy as np
import pandas as pd
import altair as alt
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords")


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/u251664/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

1. As a first step, you must pre-process the documents. In particular, for the text fields (title, 
description) you should:  

● Removing stop words 

● Tokenization 

● Removing punctuation marks 

● Stemming 

● and... anything else you think it's needed (bonus point) 

In [14]:
docs_path = "../../data/fashion_products_dataset.json"

with open(docs_path, "r", encoding="utf-8") as fp:
    data = json.load(fp)

df = pd.DataFrame(data)
print("Total number of products in the corpus: {}".format(len(df)))

Total number of products in the corpus: 28080


In [36]:
df.head()

,_id,actual_price,average_rating,brand,category,crawled_at,description,discount,images,out_of_stock,...,sub_category,title,url,title_terms,description_terms,product_details_text,details_terms,brand_terms,seller_terms,category_terms
0,fa8e22d6-c0b6-5229-bb9e-ad52eda39a0a,2999.0,3.9,York,Clothing and Accessories,1612987911000,Yorker trackpants made from 100% rich combed c...,69.0,[https://rukminim1.flixcart.com/image/128/128/...,False,...,Bottomwear,Solid Women Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Style Code 1005COMBO2 Closure Elastic Pockets ...,"[style, code, 1005combo2, closur, elast, pocke...",[york],"[shyam, enterpris]","[cloth, accessori, bottomwear]"
1,893e6980-f2a0-531f-b056-34dd63fe912c,1499.0,3.9,York,Clothing and Accessories,1612987912000,Yorker trackpants made from 100% rich combed c...,66.0,[https://rukminim1.flixcart.com/image/128/128/...,False,...,Bottomwear,Solid Men Blue Track Pants,https://www.flipkart.com/yorker-solid-men-blue...,"[solid, men, blue, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...","Style Code 1005BLUE Closure Drawstring, Elasti...","[style, code, 1005blue, closur, drawstr, elast...",[york],"[shyam, enterpris]","[cloth, accessori, bottomwear]"
2,eb4c8eab-8206-59d0-bcd1-a724d96bf74f,2999.0,3.9,York,Clothing and Accessories,1612987912000,Yorker trackpants made from 100% rich combed c...,68.0,[https://rukminim1.flixcart.com/image/128/128/...,False,...,Bottomwear,Solid Men Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, men, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Style Code 1005COMBO4 Closure Elastic Pockets ...,"[style, code, 1005combo4, closur, elast, pocke...",[york],"[shyam, enterpris]","[cloth, accessori, bottomwear]"
3,3f3f97bb-5faf-57df-a9ff-1af24e2b1045,2999.0,3.9,York,Clothing and Accessories,1612987913000,Yorker trackpants made from 100% rich combed c...,69.0,[https://rukminim1.flixcart.com/image/128/128/...,False,...,Bottomwear,Solid Women Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-mult...,"[solid, women, multicolor, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...",Style Code 1005COMBO3 Closure Elastic Pockets ...,"[style, code, 1005combo3, closur, elast, pocke...",[york],"[shyam, enterpris]","[cloth, accessori, bottomwear]"
4,750caa3d-6264-53ca-8ce1-94118a1d8951,2999.0,3.9,York,Clothing and Accessories,1612987913000,Yorker trackpants made from 100% rich combed c...,68.0,[https://rukminim1.flixcart.com/image/128/128/...,False,...,Bottomwear,"Solid Women Brown, Grey Track Pants",https://www.flipkart.com/yorker-solid-men-brow...,"[solid, women, brown, grey, track, pant]","[yorker, trackpant, made, 100, rich, comb, cot...","Style Code 1005COMBO1 Closure Drawstring, Elas...","[style, code, 1005combo1, closur, drawstr, ela...",[york],"[shyam, enterpris]","[cloth, accessori, bottomwear]"


In [ ]:
#We have collected this code from Practice 1, doing some minor changes and adding the bonus

#This functions takes a string from a focument and returns a normalised list of tokens by lowercasing, removing punctuation, tokenising, dropping, and applying  stemming.

stemmer = PorterStemmer()                
stop_words = set(stopwords.words("english"))
extra_stop_words = set() # bonus

def build_terms(line):
    """
    Preprocess the article text (title + body): removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    Argument:
    line -- string (text) to be preprocessed

    Returns:
    line - a list of tokens corresponding to the input text after the preprocessing
    """


    if not isinstance(line, str):                      # missing values (None / NaN)
        return []
    line = line.lower()                                # lowercase
    line = re.sub(r"[^a-z0-9\s]", " ", line)           # remove punctuation marks
    line = line.split()                                # Tokenize the text to get a list of terms
    line = [x for x in line if x not in stop_words and x not in extra_stop_words]
    line = [stemmer.stem(word) for word in line]       #  perform stemming
    return line

In [ ]:
#Apply the function build_terms in the fields of title and description
df["title_terms"] = [build_terms(title) for title in df["title"]]
df["description_terms"] = [build_terms(description) for description in df["description"]]

In [ ]:
# See how often every token appears and show the 30 most frequent.
title_counts = Counter(t for terms in df["title_terms"] for t in terms)
print(title_counts.most_common(30))

term_counts = Counter(t for terms in df["description_terms"] for t in terms)
print(term_counts.most_common(30))

[('shirt', 15838), ('women', 13146), ('men', 13060), ('neck', 11952), ('solid', 9246), ('round', 8301), ('print', 8123), ('pack', 4839), ('blue', 4378), ('fit', 3923), ('black', 3503), ('casual', 2927), ('slim', 2893), ('polo', 2668), ('collar', 2427), ('multicolor', 2373), ('white', 2175), ('sleev', 2030), ('full', 2011), ('2', 1966), ('grey', 1917), ('regular', 1778), ('cotton', 1656), ('3', 1568), ('sweatshirt', 1343), ('dark', 1340), ('design', 1329), ('self', 1325), ('green', 1298), ('color', 1295)]
[('shirt', 18321), ('cotton', 10139), ('wear', 9353), ('comfort', 8836), ('look', 6413), ('casual', 6034), ('fit', 5831), ('made', 5643), ('fabric', 5516), ('print', 4888), ('women', 4424), ('design', 4301), ('sleev', 4221), ('qualiti', 4087), ('men', 3775), ('wash', 3731), ('day', 3617), ('style', 3275), ('make', 3233), ('100', 3210), ('feel', 3160), ('neck', 2991), ('perfect', 2973), ('soft', 2742), ('great', 2740), ('pair', 2734), ('stylish', 2716), ('keep', 2669), ('polo', 2623), (

2. Take into account that for future queries, the final output must return (when present) the 
following information for each of the selected documents:  
pid, title, description,  brand,  category, sub_category, product_details, seller, out_of_stock, 
selling_price, discount, actual_price, average_rating, url 

In [ ]:
output_fields = ["pid", "title", "description", "brand", "category", "sub_category", "product_details", "seller", "out_of_stock", 
"selling_price", "discount", "actual_price", "average_rating", "url"]

Missing output fields: []


4. Consider the fields out_of_stock, selling_price, discount, actual_price, and average_rating. 
Decide how these should be handled during pre-processing to use in further search. Should 
they be indexed as textual terms?

In [ ]:
def parse_number(x):
    """
    Extract the first number from a string such as "2,999" or "69% off".

    Argument:
    x -- string (or number) to parse

    Returns:
    float, or NaN if no number can be extracted
    """
    #If number doesn't need pre-processing
    if isinstance(x, (int, float)):
        return float(x)

    
    match = re.search(r"\d[\d,]*\.?\d*", str(x)) # scan for the pattern anywhere
    return float(match.group().replace(",", "")) if match else np.nan #convert or fall back to NaN

df["selling_price"] = df["selling_price"].apply(parse_number)
df["actual_price"] = df["actual_price"].apply(parse_number)
df["discount"] = df["discount"].apply(parse_number)
df["average_rating"] = df["average_rating"].apply(parse_number)
df["out_of_stock"] = df["out_of_stock"].astype(bool)

print("\nMissing after parsing:\n{}".format(df[["selling_price", "actual_price", "discount", "average_rating"]].isna().sum()))

       selling_price  actual_price      discount  average_rating
count   28078.000000  27303.000000  27225.000000    25819.000000
mean      705.635088   1455.528110     50.256896        3.627724
std       549.681489    939.977456     16.887287        0.663429
min        99.000000    150.000000      1.000000        1.000000
25%       390.000000    849.000000     40.000000        3.200000
50%       545.000000   1199.000000     53.000000        3.800000
75%       820.000000   1799.000000     63.000000        4.100000
max      7999.000000  12999.000000     87.000000        5.000000

Missing after parsing:
selling_price        2
actual_price       777
discount           855
average_rating    2261
dtype: int64

out_of_stock:
out_of_stock
False    26436
True      1644
Name: count, dtype: int64

selling_price <= actual_price in 97.2% of rows


In [ ]:
#mirar segons el q escribim en Q3
LABELS_PATH = "../../data/validation_labels.csv"
labels = pd.read_csv(LABELS_PATH)

print("Labels' pids not found in df: {}".format((~labels["pid"].isin(df["pid"])).sum()))   # should be 0

queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
fields = ["title_terms", "description_terms", "details_terms", "brand_terms", "category_terms"]

relevant = labels[labels["labels"] == 1].merge(df, on="pid", how="left", suffixes=("_lab", ""))

for qid, q in queries.items():
    q_terms = set(build_terms(q))
    rel = relevant[relevant["query_id"] == qid]
    print("\nQuery {}: {} -> {}  ({} relevant products)".format(qid, q, sorted(q_terms), len(rel)))
    for f in fields:
        share = rel[f].apply(lambda terms: len(q_terms & set(terms)) / len(q_terms)).mean()
        print("  {:<18} avg share of query terms present: {:.0%}".format(f, share))

Labels' pids not found in df: 0

Query 1: women full sleeve sweatshirt cotton -> ['cotton', 'full', 'sleev', 'sweatshirt', 'women']  (13 relevant products)
  title_terms        avg share of query terms present: 80%
  description_terms  avg share of query terms present: 58%
  details_terms      avg share of query terms present: 80%
  brand_terms        avg share of query terms present: 0%
  category_terms     avg share of query terms present: 0%

Query 2: men slim jeans blue -> ['blue', 'jean', 'men', 'slim']  (10 relevant products)
  title_terms        avg share of query terms present: 80%
  description_terms  avg share of query terms present: 35%
  details_terms      avg share of query terms present: 65%
  brand_terms        avg share of query terms present: 0%
  category_terms     avg share of query terms present: 0%
